In [1]:
# ==========================================
# Task 03 - End-to-End Machine Learning Project
# ==========================================

import joblib
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings('ignore')

# ==========================================
# PART 1 - Data Preparation
# ==========================================
print('--- PART 1: Data Preparation ---')

# Note: For your Kaggle dataset (Minimum 3,000 rows), use: df = pd.read_csv('your_dataset.csv')
from sklearn.datasets import make_classification

X_raw, y_raw = make_classification(
    n_samples=4000, n_features=15, n_informative=10, random_state=42
)
feature_names = [f'feature_{i+1}' for i in range(15)]
df = pd.DataFrame(X_raw, columns=feature_names)
df['target'] = y_raw

print(f'Dataset loaded successfully with shape: {df.shape}')

# Handle missing values & duplicates
df.fillna(df.median(), inplace=True)
initial_len = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicates removed: {initial_len - len(df)}')

# Feature Engineering & Selection
X = df.drop(columns=['target'])
y = df['target']

# Data Scaling[cite: 5]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print(
    f'Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}\n'
)


# ==========================================
# PART 2 - Model Development (4 Models)
# ==========================================
print('--- PART 2: Model Development ---')

# Training at least four different machine learning models[cite: 5]
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Support Vector Machine': SVC(probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}

model_performances = {}

for name, model in models.items():
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  acc = accuracy_score(y_test, y_pred)
  model_performances[name] = acc
  print(f'{name} Accuracy: {acc:.4f}')

print('\n')


# ==========================================
# PART 3 - Model Optimization & Analysis
# ==========================================
print('--- PART 3: Model Optimization & Analysis ---')

# Cross-Validation[cite: 5]
rf = RandomForestClassifier(random_state=42)
cv_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy')
print(f'Random Forest 5-Fold CV Accuracy: {cv_scores.mean():.4f}')

# Hyperparameter Tuning using GridSearchCV[cite: 5]
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5],
}
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
print('Best Hyperparameters:', grid_search.best_params_)

# Feature Importance Analysis[cite: 5]
importances = best_model.feature_importances_
feature_imp_df = pd.DataFrame(
    {'Feature': X.columns, 'Importance': importances}
).sort_values(by='Importance', ascending=False)
print('\nTop 5 Important Features:')
print(feature_imp_df.head(), '\n')


# ==========================================
# PART 4 - Save the Best Model
# ==========================================
print('--- PART 4: Save the Best Model ---')

# Saving the best model using Joblib
model_filename = 'best_model.pkl'
joblib.dump(best_model, model_filename)
print(f'Best model successfully saved as {model_filename} using Joblib!\n')


--- PART 1: Data Preparation ---
Dataset loaded successfully with shape: (4000, 16)
Duplicates removed: 0
Training set shape: (3200, 15), Testing set shape: (800, 15)

--- PART 2: Model Development ---
Decision Tree Accuracy: 0.9025
Random Forest Accuracy: 0.9500
Support Vector Machine Accuracy: 0.9575
Gradient Boosting Accuracy: 0.9313


--- PART 3: Model Optimization & Analysis ---
Random Forest 5-Fold CV Accuracy: 0.9497
Best Hyperparameters: {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 100}

Top 5 Important Features:
       Feature  Importance
1    feature_2    0.147417
5    feature_6    0.124045
11  feature_12    0.107510
10  feature_11    0.085368
3    feature_4    0.077107 

--- PART 4: Save the Best Model ---
Best model successfully saved as best_model.pkl using Joblib!

